
# Experimentación: reconocimiento de entidades nombradas (NER)

## 1. Objetivo

Evaluar modelos preentrenados de reconocimiento de entidades nombradas para identificar personas, organizaciones y lugares mencionados en noticias digitales.

Se priorizan modelos multilingües que permitan procesar textos en español con presencia de nombres propios y términos en inglés.

## 2. Primer modelo: XLM-RoBERTa NER

Se utiliza el modelo `Davlan/xlm-roberta-base-ner-hrl`, basado en la arquitectura XLM-RoBERTa y ajustado para reconocimiento de entidades nombradas.

El experimento comprende:

1. Carga del modelo preentrenado.
2. Pruebas sobre textos controlados.
3. Procesamiento de noticias reales.
4. Inspección de entidades y errores.
5. Comparación posterior con GLiNER.

La inferencia se ejecuta localmente.


In [6]:
import pandas as pd
import numpy as np
import psycopg
import os
import logging
from transformers import pipeline

In [7]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [1]:
%pip install transformers torch pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import logging

import pandas as pd
from transformers import pipeline


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

logger = logging.getLogger(__name__)

NER_CONFIG = {
    "model_name": "Davlan/xlm-roberta-base-ner-hrl",
    "aggregation_strategy": "simple",
    "device": -1,
}


In [3]:

def load_ner_model(config):
    """
    Carga un modelo Transformer para reconocimiento de entidades.

    Parameters
    ----------
    config : dict
        Configuración del modelo NER.

    Returns
    -------
    transformers.pipelines.TokenClassificationPipeline
        Pipeline de reconocimiento de entidades.
    """
    logger.info(
        "Cargando modelo NER: %s",
        config["model_name"],
    )

    classifier = pipeline(
        task="token-classification",
        model=config["model_name"],
        tokenizer=config["model_name"],
        aggregation_strategy=config["aggregation_strategy"],
        device=config["device"],
    )

    logger.info("Modelo NER cargado correctamente.")

    return classifier


ner_classifier = load_ner_model(NER_CONFIG)


2026-10-26 21:05:24,694 - INFO - Cargando modelo NER: Davlan/xlm-roberta-base-ner-hrl
2026-10-26 21:05:24,984 - INFO - HTTP Request: GET https://huggingface.co/api/models/Davlan/xlm-roberta-base-ner-hrl "HTTP/1.1 200 OK"
2026-10-26 21:05:25,167 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-26 21:05:25,330 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Davlan/xlm-roberta-base-ner-hrl/253f557bd8249b8515114cfd7f71974fe5fa4d2f/config.json?%2FDavlan%2Fxlm-roberta-base-ner-hrl%2Fresolve%2F253f557bd8249b8515114cfd7f71974fe5fa4d2f%2Fconfig.json=&etag=%22408f9ee1f8a09dacfec2dd20fbf55f1f276a2642%22 "HTTP/1.1 200 OK"
2026-10-26 21:05:25,499 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Davlan/xlm-roberta-base-ner-hrl/253f557bd8249b8515114cfd7f71974fe5fa4d2f/config.json?%2FDavlan%2Fxlm-roberta-base-ner-

config.json:   0%|          | 0.00/980 [00:00<?, ?B/s]

e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\enzor\.cache\huggingface\hub\models--Davlan--xlm-roberta-base-ner-hrl. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
2026-10-26 21:05:25,745 - INFO - HTTP Request: HEAD https://huggingface.c

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (4 outputs left out)

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

2026-10-26 21:05:52,030 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/tokenizer.json "HTTP/1.1 404 Not Found"
2026-10-26 21:05:52,216 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/added_tokens.json "HTTP/1.1 404 Not Found"
2026-10-26 21:05:52,394 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-10-26 21:05:52,558 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Davlan/xlm-roberta-base-ner-hrl/253f557bd8249b8515114cfd7f71974fe5fa4d2f/special_tokens_map.json?%2FDavlan%2Fxlm-roberta-base-ner-hrl%2Fresolve%2F253f557bd8249b8515114cfd7f71974fe5fa4d2f%2Fspecial_tokens_map.json=&etag=%226cd1d9021e10d47aed59399af6b0e30312b46ca4%22 "HTTP/1.1 200 OK"
2026-10

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

2026-10-26 21:05:52,926 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-10-26 21:05:53,683 - INFO - HTTP Request: GET https://huggingface.co/api/models/Davlan/xlm-roberta-base-ner-hrl "HTTP/1.1 200 OK"
2026-10-26 21:05:53,861 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/processor_config.json "HTTP/1.1 404 Not Found"
2026-10-26 21:05:54,040 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-10-26 21:05:54,224 - INFO - HTTP Request: HEAD https://huggingface.co/Davlan/xlm-roberta-base-ner-hrl/resolve/253f557bd8249b8515114cfd7f71974fe5fa4d2f/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-10-26 21:05:54,226 - INFO - Modelo NER

In [4]:

test_text = (
    "Sam Altman, director ejecutivo de OpenAI, anunció "
    "nuevas inversiones en inteligencia artificial "
    "durante una conferencia en San Francisco. "
    "Microsoft y Google también participaron del evento."
)

entities = ner_classifier(test_text)

test_results = pd.DataFrame(
    [
        {
            "entity": entity["word"],
            "type": entity["entity_group"],
            "score": round(float(entity["score"]), 4),
            "start": entity["start"],
            "end": entity["end"],
        }
        for entity in entities
    ]
)

display(test_results)


,entity,type,score,start,end
0,Sam Altman,PER,0.9998,0,10
1,OpenAI,ORG,0.9990,34,40
2,San Francisco,LOC,0.9996,123,136
3,Microsoft,ORG,0.9998,138,147
4,Google,ORG,0.9998,150,156



## 3. Experimentación con noticias reales

Se evalúa el reconocimiento de entidades sobre las noticias almacenadas en PostgreSQL, utilizando el título y la descripción como entrada.

El objetivo es observar las entidades detectadas, sus categorías y las puntuaciones de confianza, identificando posibles errores y limitaciones del modelo.


In [8]:

def get_news_for_ner(connection):
    """Recupera las noticias para experimentar con NER."""
    query = """
        SELECT
            rss_news_id,
            title,
            description
        FROM rss_news
        ORDER BY rss_news_id;
    """

    return pd.read_sql_query(query, connection)


connection = get_database_connection()

try:
    news_df = get_news_for_ner(connection)
finally:
    connection.close()

logger.info("Noticias recuperadas: %s", len(news_df))

display(news_df.head())


C:\Users\enzor\AppData\Local\Temp\ipykernel_17072\368711466.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, connection)
2026-10-26 21:07:15,503 - INFO - Noticias recuperadas: 186


,rss_news_id,title,description
0,1,Guía para obtener 15 GB de espacio gratis en G...,"Videos, respaldos antiguos e instaladores suel..."
1,2,Epic Games Store regala un nuevo juego de terr...,La tienda suma un indie de supervivencia y exp...
2,3,Bill Gates advierte sobre la IA: “será el mayo...,"Según el fundador de Microsoft, esta tecnologí..."
3,4,WhatsApp refuerza la seguridad de las cuentas ...,La nueva función busca combatir las tácticas d...
4,5,Gratis en Steam: el juego que combina superviv...,El juego mezcla supervivencia con dinosaurios ...


In [9]:

def build_ner_text(row):
    """Combina título y descripción de una noticia."""
    title = str(row["title"]) if pd.notna(row["title"]) else ""
    description = (
        str(row["description"])
        if pd.notna(row["description"])
        else ""
    )

    return ". ".join(
        part.strip()
        for part in [title, description]
        if part.strip()
    )


news_df["ner_text"] = news_df.apply(
    build_ner_text,
    axis=1,
)

display(news_df[["rss_news_id", "ner_text"]].head())


,rss_news_id,ner_text
0,1,Guía para obtener 15 GB de espacio gratis en G...
1,2,Epic Games Store regala un nuevo juego de terr...
2,3,Bill Gates advierte sobre la IA: “será el mayo...
3,4,WhatsApp refuerza la seguridad de las cuentas ...
4,5,Gratis en Steam: el juego que combina superviv...



## 4. Extracción de entidades nombradas

Se aplica el modelo NER sobre los títulos y descripciones de las noticias, procesando los textos en lotes.

Para cada entidad identificada se registra la noticia de origen, el texto de la entidad, su categoría, la puntuación de confianza y su posición dentro del texto.

Los resultados se organizan en un DataFrame para facilitar su exploración y posterior evaluación.


In [11]:

def extract_news_entities(news, classifier, batch_size=8):
    """
    Extrae entidades nombradas de un conjunto de noticias.

    Parameters
    ----------
    news : pandas.DataFrame
        Noticias con identificador y texto preparado.
    classifier : transformers.Pipeline
        Modelo NER inicializado.
    batch_size : int
        Cantidad de textos procesados por lote.

    Returns
    -------
    pandas.DataFrame
        Entidades identificadas por noticia.
    """
    valid_news = news.loc[
        news["ner_text"].str.strip().ne("")
    ].copy()

    texts = valid_news["ner_text"].tolist()

    predictions = classifier(
        texts,
        batch_size=batch_size,
    )

    records = []

    for (_, row), entities in zip(
        valid_news.iterrows(),
        predictions,
    ):
        for entity in entities:
            records.append(
                {
                    "rss_news_id": int(row["rss_news_id"]),
                    "entity_text": entity["word"],
                    "entity_type": entity["entity_group"],
                    "confidence": float(entity["score"]),
                    "start_char": int(entity["start"]),
                    "end_char": int(entity["end"]),
                }
            )

    return pd.DataFrame(
        records,
        columns=[
            "rss_news_id",
            "entity_text",
            "entity_type",
            "confidence",
            "start_char",
            "end_char",
        ],
    )


ner_results = extract_news_entities(
    news=news_df,
    classifier=ner_classifier,
    batch_size=8,
)

logger.info(
    "Entidades identificadas: %s",
    len(ner_results),
)

display(ner_results.head(20))


2026-10-26 21:08:48,184 - INFO - Entidades identificadas: 348


,rss_news_id,entity_text,entity_type,confidence,start_char,end_char
0,1,Google Drive,ORG,0.988421,45,57
1,1,Gmail,ORG,0.998026,59,64
2,1,Google Fotos,ORG,0.981861,67,79
3,2,Epic Games Store,ORG,0.999817,0,16
4,3,Bill Gates,PER,0.999844,0,10
5,3,Microsoft,ORG,0.999556,128,137
6,4,WhatsApp,ORG,0.991371,0,8
7,5,Steam,ORG,0.663446,10,15
8,7,Time,ORG,0.998246,0,4
9,7,IA,ORG,0.986518,57,59


In [12]:

print(f"Noticias analizadas: {len(news_df)}")
print(f"Entidades identificadas: {len(ner_results)}")
print(
    "Noticias con entidades:",
    ner_results["rss_news_id"].nunique(),
)

display(
    ner_results["entity_type"]
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="count")
)


Noticias analizadas: 186
Entidades identificadas: 348
Noticias con entidades: 135


,entity_type,count
0,ORG,224
1,LOC,70
2,PER,54



## 5. Inspección cualitativa de resultados

Se selecciona una muestra reproducible de noticias para examinar las entidades identificadas por XLM-RoBERTa.

La inspección busca detectar:

- Entidades identificadas correctamente.
- Falsos positivos.
- Entidades omitidas.
- Errores en la categoría asignada.
- Entidades incompletas o fragmentadas.

Esta etapa es exploratoria y no constituye todavía una evaluación cuantitativa del rendimiento del modelo.


In [13]:

entity_counts = (
    ner_results.groupby("rss_news_id")
    .size()
    .rename("entity_count")
)

news_sample_base = news_df.merge(
    entity_counts,
    left_on="rss_news_id",
    right_index=True,
    how="left",
)

news_sample_base["entity_count"] = (
    news_sample_base["entity_count"]
    .fillna(0)
    .astype(int)
)

sample_parts = []

for condition in [
    news_sample_base["entity_count"].eq(0),
    news_sample_base["entity_count"].between(1, 2),
    news_sample_base["entity_count"].ge(3),
]:
    group = news_sample_base.loc[condition]

    sample_parts.append(
        group.sample(
            n=min(5, len(group)),
            random_state=42,
        )
    )

ner_sample = (
    pd.concat(sample_parts)
    .sort_values("rss_news_id")
    .reset_index(drop=True)
)

print(f"Noticias seleccionadas: {len(ner_sample)}")

display(
    ner_sample[
        ["rss_news_id", "title", "entity_count"]
    ]
)


Noticias seleccionadas: 15


,rss_news_id,title,entity_count
0,1,Guía para obtener 15 GB de espacio gratis en G...,3
1,2,Epic Games Store regala un nuevo juego de terr...,1
2,23,Por qué Londres está frenando la llegada de lo...,3
3,37,Adiós a las redes sociales: cómo configurar An...,0
4,39,"La IA avanza en Perú, pero las empresas enfren...",2
5,45,PlayStation Plus confirma juegos gratis para s...,2
6,64,Riesgos de ver el último capítulo de Cien años...,4
7,65,Meta debe cambiar de inmediato las reglas para...,1
8,66,Elon Musk y Mark Zuckerberg creen que el celul...,2
9,81,El peligro de dejar activos el WiFi y Bluetoot...,0


In [14]:

def inspect_ner_sample(sample, entities):
    """Muestra el texto original y las entidades detectadas."""
    for _, news in sample.iterrows():
        news_id = news["rss_news_id"]

        detected = entities.loc[
            entities["rss_news_id"].eq(news_id),
            ["entity_text", "entity_type", "confidence"],
        ]

        print("=" * 80)
        print(f"NOTICIA ID: {news_id}")
        print(f"TEXTO: {news['ner_text']}")
        print("\nENTIDADES DETECTADAS:")

        if detected.empty:
            print("Ninguna")
        else:
            print(
                detected.to_string(index=False)
            )

        print()


inspect_ner_sample(
    sample=ner_sample,
    entities=ner_results,
)


NOTICIA ID: 1
TEXTO: Guía para obtener 15 GB de espacio gratis en Google Drive, Gmail y Google Fotos. Videos, respaldos antiguos e instaladores suelen estar entre los principales responsables, y al borrarlos y vaciar la papelera se recupera almacenamiento efectivo

ENTIDADES DETECTADAS:
 entity_text entity_type  confidence
Google Drive         ORG    0.988421
       Gmail         ORG    0.998026
Google Fotos         ORG    0.981861

NOTICIA ID: 2
TEXTO: Epic Games Store regala un nuevo juego de terror cooperativo para PC. La tienda suma un indie de supervivencia y exploración en un bosque paranormal, con énfasis en comunicación en equipo y tensión atmosférica

ENTIDADES DETECTADAS:
     entity_text entity_type  confidence
Epic Games Store         ORG    0.999817

NOTICIA ID: 23
TEXTO: Por qué Londres está frenando la llegada de los robotaxis a pesar de que Uber ya tiene la licencia. El tráfico en la ciudad es uno de los factores que ha impedido este avance, que ya está presente en otro


## 6. Experimentación con GLiNER

Se evalúa GLiNER Multilingual v2.1, un modelo de reconocimiento de entidades nombradas que permite especificar las categorías de interés durante la inferencia.

A diferencia del modelo XLM-RoBERTa NER utilizado anteriormente, GLiNER permite ampliar las categorías de extracción sin realizar un nuevo entrenamiento.

Se consideran cinco categorías:

- `person`: personas.
- `organization`: organizaciones.
- `location`: lugares.
- `product`: productos y servicios.
- `technology`: tecnologías y conceptos tecnológicos.

Inicialmente se realiza una prueba controlada para verificar el funcionamiento del modelo y observar su capacidad de distinguir entre organizaciones, productos y tecnologías.


In [15]:
%pip install gliner

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/262.0 kB ? eta -:--:--
   ---------------------------------------  256.0/262.0 kB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 262.0/262.0 kB 5.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.1 MB 10.0 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/12.1 MB 10.4 MB/s eta 0:00:02
   ----- ---------------------------------- 1.6/12.1 MB 11.6 MB/s eta 0:00:01
   ------ --------------------------------- 2.1/12.1 MB 12.2 MB/s eta 0:00:01
   ------ --------------------------------- 2.1/12.1 MB 12.2 MB/s eta 0:00:01
   ------ --------------------------------- 2.1/12.1 MB 12.2 MB/s eta 0:00:01
   -------- --------------------


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:

from gliner import GLiNER


GLINER_CONFIG = {
    "model_name": "urchade/gliner_multi-v2.1",
    "threshold": 0.5,
}

ENTITY_LABELS = [
    "person",
    "organization",
    "location",
    "product",
]


def load_gliner_model(config):
    """Carga el modelo multilingüe GLiNER."""
    logger.info(
        "Cargando modelo GLiNER: %s",
        config["model_name"],
    )

    model = GLiNER.from_pretrained(
        config["model_name"],
    )

    model.eval()

    logger.info("Modelo GLiNER cargado correctamente.")

    return model


gliner_model = load_gliner_model(GLINER_CONFIG)


2026-10-26 21:39:00,090 - INFO - Cargando modelo GLiNER: urchade/gliner_multi-v2.1
e:\Trabajo final UGR\communication-intelligence-assistant\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  if resume_download is not None:
2026-10-26 21:39:00,349 - INFO - HTTP Request: GET https://huggingface.co/api/models/urchade/gliner_multi-v2.1/revision/main "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

2026-10-26 21:39:00,385 - INFO - Loading the following GLiNER type: <class 'gliner.model.UniEncoderSpanGLiNER'>...
2026-10-26 21:39:00,559 - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base "HTTP/1.1 200 OK"
2026-10-26 21:39:00,740 - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-26 21:39:00,918 - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-26 21:39:01,508 - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base "HTTP/1.1 200 OK"
2026-10-26 21:39:01,683 - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base "HTTP/1.1 200 OK"
2026-10-26 21:39:05,363 - INFO - Modelo GLiNER cargado correctamente.


In [17]:

test_text_gliner = (
    "Sam Altman, director ejecutivo de OpenAI, presentó "
    "ChatGPT durante una conferencia en San Francisco. "
    "Microsoft anunció nuevas inversiones en inteligencia "
    "artificial y computación en la nube. "
    "Google también presentó novedades de Gemini."
)

gliner_entities = gliner_model.predict_entities(
    text=test_text_gliner,
    labels=ENTITY_LABELS,
    threshold=GLINER_CONFIG["threshold"],
)

gliner_test_results = pd.DataFrame(
    [
        {
            "entity": entity["text"],
            "type": entity["label"],
            "score": round(float(entity["score"]), 4),
            "start": entity["start"],
            "end": entity["end"],
        }
        for entity in gliner_entities
    ]
)

display(gliner_test_results)


,entity,type,score,start,end
0,Sam Altman,person,0.9904,0,10
1,OpenAI,organization,0.9629,34,40
2,ChatGPT,product,0.8192,51,58
3,San Francisco,location,0.9854,86,99
4,Microsoft,organization,0.9915,101,110
5,inteligencia artificial,technology,0.5861,141,164
6,computación en la nube,technology,0.5793,167,189
7,Google,organization,0.9907,191,197
8,Gemini,product,0.8016,228,234



## 7. Evaluación exploratoria sobre noticias reales

Se aplica GLiNER sobre la misma muestra de 15 noticias utilizada anteriormente con XLM-RoBERTa.

El objetivo es comparar cualitativamente la identificación de personas, organizaciones y lugares, así como analizar la capacidad adicional de GLiNER para reconocer productos, servicios y tecnologías.

Se mantiene un umbral inicial de confianza de 0,5.


In [21]:

def extract_gliner_entities(news, model, labels, threshold=0.5):
    """
    Extrae entidades nombradas de una muestra de noticias.

    Parameters
    ----------
    news : pandas.DataFrame
        Noticias seleccionadas para evaluación.
    model : GLiNER
        Modelo GLiNER inicializado.
    labels : list
        Categorías de entidades.
    threshold : float
        Umbral mínimo de confianza.

    Returns
    -------
    pandas.DataFrame
        Entidades identificadas por noticia.
    """
    records = []

    for _, row in news.iterrows():
        entities = model.predict_entities(
            text=row["ner_text"],
            labels=labels,
            threshold=threshold,
        )

        for entity in entities:
            records.append(
                {
                    "rss_news_id": int(row["rss_news_id"]),
                    "entity_text": entity["text"],
                    "entity_type": entity["label"],
                    "confidence": float(entity["score"]),
                    "start_char": int(entity["start"]),
                    "end_char": int(entity["end"]),
                }
            )

    return pd.DataFrame(
        records,
        columns=[
            "rss_news_id",
            "entity_text",
            "entity_type",
            "confidence",
            "start_char",
            "end_char",
        ],
    )


gliner_results = extract_gliner_entities(
    news=ner_sample,
    model=gliner_model,
    labels=ENTITY_LABELS,
    threshold=GLINER_CONFIG["threshold"],
)

print(f"Noticias evaluadas: {len(ner_sample)}")
print(f"Entidades identificadas: {len(gliner_results)}")

display(gliner_results)


Noticias evaluadas: 15
Entidades identificadas: 52


,rss_news_id,entity_text,entity_type,confidence,start_char,end_char
0,1,Google Drive,product,0.873393,45,57
1,1,Gmail,product,0.970584,59,64
2,1,Google Fotos,product,0.918989,67,79
3,1,Videos,product,0.748688,81,87
4,1,instaladores,product,0.567629,110,122
5,2,Epic Games Store,organization,0.973131,0,16
6,2,PC,location,0.830045,66,68
7,2,indie,product,0.559398,88,93
8,2,bosque paranormal,location,0.671713,131,148
9,23,Londres,location,0.962418,8,15


In [22]:

inspect_ner_sample(
    sample=ner_sample,
    entities=gliner_results,
)


NOTICIA ID: 1
TEXTO: Guía para obtener 15 GB de espacio gratis en Google Drive, Gmail y Google Fotos. Videos, respaldos antiguos e instaladores suelen estar entre los principales responsables, y al borrarlos y vaciar la papelera se recupera almacenamiento efectivo

ENTIDADES DETECTADAS:
 entity_text entity_type  confidence
Google Drive     product    0.873393
       Gmail     product    0.970584
Google Fotos     product    0.918989
      Videos     product    0.748688
instaladores     product    0.567629

NOTICIA ID: 2
TEXTO: Epic Games Store regala un nuevo juego de terror cooperativo para PC. La tienda suma un indie de supervivencia y exploración en un bosque paranormal, con énfasis en comunicación en equipo y tensión atmosférica

ENTIDADES DETECTADAS:
      entity_text  entity_type  confidence
 Epic Games Store organization    0.973131
               PC     location    0.830045
            indie      product    0.559398
bosque paranormal     location    0.671713

NOTICIA ID: 23
TEXT


## 8. Experimentación con un enfoque híbrido de NER

Se propone combinar XLM-RoBERTa y GLiNER para aprovechar sus capacidades complementarias.

XLM-RoBERTa se utiliza para reconocer personas, organizaciones y lugares, mientras que GLiNER se utiliza para identificar productos y servicios.

Posteriormente, se aplica una estrategia de consolidación que elimina duplicados y resuelve entidades superpuestas, dando prioridad a las predicciones de productos de GLiNER.

Este enfoque se evalúa inicialmente sobre las mismas 15 noticias utilizadas en las pruebas anteriores.


In [23]:

gliner_products = extract_gliner_entities(
    news=ner_sample,
    model=gliner_model,
    labels=["product"],
    threshold=GLINER_CONFIG["threshold"],
)

print(f"Productos detectados: {len(gliner_products)}")

display(gliner_products)


Productos detectados: 8


,rss_news_id,entity_text,entity_type,confidence,start_char,end_char
0,1,Google Drive,product,0.949450,45,57
1,1,Gmail,product,0.970322,59,64
2,1,Google Fotos,product,0.960791,67,79
3,23,Uber,product,0.661507,73,77
4,37,Android,product,0.599882,44,51
5,64,Netflix,product,0.624704,112,119
6,66,el celular,product,0.901141,38,48
7,176,cañas,product,0.715987,32,37


In [24]:

def merge_ner_predictions(xlm_results, product_results, news_ids):
    """
    Combina las entidades de XLM-RoBERTa y GLiNER.

    Prioriza productos de GLiNER cuando existe
    superposición con una entidad de XLM-RoBERTa.
    """
    xlm_entities = xlm_results[
        xlm_results["rss_news_id"].isin(news_ids)
    ].copy()

    products = product_results.copy()

    # Normalizar categorías de XLM-RoBERTa.
    xlm_entities["entity_type"] = xlm_entities[
        "entity_type"
    ].replace({
        "PER": "person",
        "ORG": "organization",
        "LOC": "location",
    })

    xlm_entities["model_source"] = "XLM-RoBERTa"
    products["model_source"] = "GLiNER"

    # Incorporar productos primero para darles prioridad.
    combined = pd.concat(
        [products, xlm_entities],
        ignore_index=True,
    )

    combined = combined.sort_values(
        ["rss_news_id", "model_source", "start_char"],
        ascending=[True, True, True],
    )

    selected = []

    for _, entity in combined.iterrows():
        overlaps = any(
            existing["rss_news_id"] == entity["rss_news_id"]
            and entity["start_char"] < existing["end_char"]
            and entity["end_char"] > existing["start_char"]
            for existing in selected
        )

        if not overlaps:
            selected.append(entity.to_dict())

    return (
        pd.DataFrame(selected, columns=combined.columns)
        .sort_values(["rss_news_id", "start_char"])
        .reset_index(drop=True)
    )


hybrid_results = merge_ner_predictions(
    xlm_results=ner_results,
    product_results=gliner_products,
    news_ids=ner_sample["rss_news_id"].tolist(),
)

print(f"Entidades híbridas identificadas: {len(hybrid_results)}")

display(hybrid_results)


Entidades híbridas identificadas: 32


,rss_news_id,entity_text,entity_type,confidence,start_char,end_char,model_source
0,1,Google Drive,product,0.949450,45,57,GLiNER
1,1,Gmail,product,0.970322,59,64,GLiNER
2,1,Google Fotos,product,0.960791,67,79,GLiNER
3,2,Epic Games Store,organization,0.999817,0,16,XLM-RoBERTa
4,23,Londres,location,0.995726,8,15,XLM-RoBERTa
5,23,Uber,product,0.661507,73,77,GLiNER
6,23,Europa,location,0.999582,216,222,XLM-RoBERTa
7,37,Android,product,0.599882,44,51,GLiNER
8,39,IA,organization,0.697308,3,5,XLM-RoBERTa
9,39,Perú,location,0.999513,16,20,XLM-RoBERTa


In [25]:

inspect_ner_sample(
    sample=ner_sample,
    entities=hybrid_results,
)


NOTICIA ID: 1
TEXTO: Guía para obtener 15 GB de espacio gratis en Google Drive, Gmail y Google Fotos. Videos, respaldos antiguos e instaladores suelen estar entre los principales responsables, y al borrarlos y vaciar la papelera se recupera almacenamiento efectivo

ENTIDADES DETECTADAS:
 entity_text entity_type  confidence
Google Drive     product    0.949450
       Gmail     product    0.970322
Google Fotos     product    0.960791

NOTICIA ID: 2
TEXTO: Epic Games Store regala un nuevo juego de terror cooperativo para PC. La tienda suma un indie de supervivencia y exploración en un bosque paranormal, con énfasis en comunicación en equipo y tensión atmosférica

ENTIDADES DETECTADAS:
     entity_text  entity_type  confidence
Epic Games Store organization    0.999817

NOTICIA ID: 23
TEXTO: Por qué Londres está frenando la llegada de los robotaxis a pesar de que Uber ya tiene la licencia. El tráfico en la ciudad es uno de los factores que ha impedido este avance, que ya está presente en ot